# جلسه ۵: تولید تقویت‌شده با بازیابی (RAG)

## اهداف
- درک معماری RAG و اهمیت آن
- پیاده‌سازی تکه‌بندی اسناد
- ساخت خط لوله کامل RAG از صفر
- پاسخ به سؤالات با استفاده از دانش سفارشی

**مدت زمان:** ۴۰ دقیقه | **سطح:** متوسط

**چرا RAG؟** LLMها فقط آنچه را که آموزش دیده‌اند می‌دانند. RAG به آنها اجازه می‌دهد بدون نیاز به تنظیم دقیق، به سؤالات درباره داده‌های شما — اسناد، پایگاه‌های داده، پایگاه‌های دانش — پاسخ دهند.

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# بارگذاری متغیرهای محیطی از فایل .env
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"راه‌اندازی کامل شد! مدل: {MODEL}")

Setup complete! Model: gpt-5-mini


## ۱. معماری RAG

RAG دو فاز اصلی دارد:

### فاز ایندکس‌گذاری (یک بار انجام می‌شود)
```
اسناد → تکه‌بندی → امبد → ذخیره در ایندکس برداری
```

### فاز پرسش (به ازای هر سؤال)
```
سؤال کاربر → امبد → جستجوی ایندکس → بازیابی تکه‌های برتر → تغذیه به LLM → پاسخ
```

نکته کلیدی: ما زمینه مرتبط را **بازیابی** و پرامپت LLM را با آن **تقویت** می‌کنیم.

## ۲. آماده‌سازی اسناد: تکه‌بندی

اسناد طولانی باید به **تکه‌های** کوچکتر تقسیم شوند زیرا:
- امبدینگ‌ها روی متن متمرکز بهتر کار می‌کنند
- LLMها پنجره زمینه محدودی دارند
- بازیابی با تکه‌های کوچکتر دقیق‌تر است

In [ ]:
# سند نمونه — تصور کنید این پایگاه دانش داخلی یک شرکت است
document = """
# Company Policies and Guidelines

## Remote Work Policy
Employees may work remotely up to 3 days per week. Remote work days must be agreed upon with your manager. All remote workers must be available during core hours (10 AM - 3 PM). A stable internet connection is required for remote work.

## Leave Policy
Full-time employees receive 20 days of paid annual leave. Sick leave is provided at 10 days per year. Leave requests must be submitted at least 2 weeks in advance for planned leave. Parental leave is 12 weeks for primary caregivers and 4 weeks for secondary caregivers.

## Expense Policy
Business expenses must be submitted within 30 days of the expense. Meals during business travel are reimbursed up to $75 per day. Flights must be booked economy class for trips under 6 hours. Hotel accommodations should not exceed $200 per night without manager approval.

## Professional Development
Each employee has an annual learning budget of $2,000. This can be used for courses, conferences, books, and certifications. Requests must be approved by your manager before purchase. Time spent on approved learning during work hours is considered work time.

## IT Security
All company devices must use full disk encryption. Passwords must be at least 12 characters with a mix of letters, numbers, and symbols. Two-factor authentication is required for all company accounts. Report any security incidents to security@company.com immediately.
"""

print(f"طول سند: {len(document)} کاراکتر")

Document length: 1444 characters


In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    """تقسیم متن به تکه‌های همپوشان.
    
    آرگومان‌ها:
        text: متنی که باید تکه‌بندی شود
        chunk_size: تعداد تقریبی کاراکترها در هر تکه
        overlap: کاراکترهای همپوشان بین تکه‌ها (تداوم زمینه را فراهم می‌کند)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        # سعی در شکستن در مرز جمله
        if end < len(text):
            # جستجوی آخرین نقطه یا خط جدید نزدیک انتها
            break_point = text.rfind('.', start, end)
            if break_point == -1:
                break_point = text.rfind('\n', start, end)
            if break_point > start:
                end = break_point + 1
        
        chunk = text[start:end].strip()
        if chunk:  # رد شدن از تکه‌های خالی
            chunks.append(chunk)
        start = end - overlap
    
    return chunks

# تکه‌بندی سند
chunks = chunk_text(document, chunk_size=300, overlap=50)

print(f"{len(chunks)} تکه ایجاد شد\n")
for i, chunk in enumerate(chunks):
    print(f"--- تکه {i} ({len(chunk)} کاراکتر) ---")
    print(chunk[:100] + "...")
    print()

Created 7 chunks

--- Chunk 0 (292 chars) ---
# Company Policies and Guidelines

## Remote Work Policy
Employees may work remotely up to 3 days pe...

--- Chunk 1 (249 chars) ---
e internet connection is required for remote work.

## Leave Policy
Full-time employees receive 20 d...

--- Chunk 2 (287 chars) ---
ted at least 2 weeks in advance for planned leave. Parental leave is 12 weeks for primary caregivers...

--- Chunk 3 (275 chars) ---
business travel are reimbursed up to $75 per day. Flights must be booked economy class for trips und...

--- Chunk 4 (253 chars) ---
employee has an annual learning budget of $2,000. This can be used for courses, conferences, books, ...

--- Chunk 5 (267 chars) ---
earning during work hours is considered work time.

## IT Security
All company devices must use full...

--- Chunk 6 (117 chars) ---
thentication is required for all company accounts. Report any security incidents to security@company...



## ۳. ساخت خط لوله RAG

حالا بیایید همه چیز را کنار هم بگذاریم: تکه‌بندی → امبد → جستجو → تولید.

In [ ]:
class SimpleRAG:
    """خط لوله RAG ساده با استفاده از امبدینگ‌ها و chat completions اوپن‌ای‌آی."""
    
    def __init__(self, chunk_size=300, overlap=50):
        self.chunks = []
        self.embeddings = []
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def add_document(self, text):
        """افزودن سند به پایگاه دانش."""
        new_chunks = chunk_text(text, self.chunk_size, self.overlap)
        new_embeddings = get_embeddings(new_chunks)
        self.chunks.extend(new_chunks)
        self.embeddings.extend(new_embeddings)
        print(f"{len(new_chunks)} تکه اضافه شد (کل: {len(self.chunks)})")
    
    def retrieve(self, query, top_k=3):
        """یافتن مرتبط‌ترین تکه‌ها برای یک پرسش."""
        query_embedding = get_embedding(query)
        
        similarities = [
            cosine_similarity(query_embedding, emb)
            for emb in self.embeddings
        ]
        
        scored = list(zip(similarities, self.chunks))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]
    
    def ask(self, question, top_k=3):
        """پاسخ به سؤال با استفاده از زمینه بازیابی‌شده."""
        # مرحله ۱: بازیابی تکه‌های مرتبط
        results = self.retrieve(question, top_k)
        
        # مرحله ۲: ساخت زمینه از تکه‌های بازیابی‌شده
        context = "\n\n".join([chunk for _, chunk in results])
        
        # مرحله ۳: تولید پاسخ با LLM و زمینه
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": """You are a helpful assistant that answers questions based on the provided context.
Only answer based on the context given. If the context doesn't contain the answer, say "I don't have enough information to answer that."""},
                {"role": "user", "content": f"""Context:\n{context}\n\nQuestion: {question}"""}
            ],
            temperature=0
        )
        
        answer = response.choices[0].message.content
        return answer, results

print("کلاس SimpleRAG تعریف شد!")

SimpleRAG class defined!


In [ ]:
# راه‌اندازی RAG و افزودن سند
rag = SimpleRAG(chunk_size=300, overlap=50)
rag.add_document(document)

NameError: name 'get_embeddings' is not defined

In [ ]:
# پرسیدن سؤال درباره سند
question = "How many days can I work from home?"
answer, sources = rag.ask(question)

print(f"سؤال: {question}")
print(f"پاسخ: {answer}")
print(f"\n--- منابع (بهترین تطبیق) ---")
print(f"[{sources[0][0]:.4f}] {sources[0][1][:100]}...")

NameError: name 'get_embedding' is not defined

In [ ]:
# سؤالات بیشتر
questions = [
    "What is the budget for professional development?",
    "How much can I spend on dinner during a business trip?",
    "What are the password requirements?",
    "How long is parental leave?"
]

for q in questions:
    answer, _ = rag.ask(q)
    print(f"سؤال: {q}")
    print(f"پاسخ: {answer}\n")

NameError: name 'get_embedding' is not defined

## ۴. تست مرزهای RAG

یک سیستم RAG خوب باید وقتی پاسخ در اسناد نیست بگوید «نمی‌دانم».

In [ ]:
# سؤالی بپرسید که در سند نیست
answer, sources = rag.ask("What is the company's stock price?")
print(f"سؤال: What is the company's stock price?")
print(f"پاسخ: {answer}")
print(f"\nامتیاز بهترین تطبیق: {sources[0][0]:.4f}")
# توجه: مدل به درستی می‌گوید این اطلاعات را ندارد

NameError: name 'get_embedding' is not defined

## ۵. افزودن اسناد بیشتر

RAG به راحتی مقیاس‌پذیر است — فقط اسناد بیشتری به ایندکس اضافه کنید.

In [ ]:
# افزودن سند دوم
tech_doc = """
## Tech Stack Documentation

Our backend services use Python with FastAPI framework. The primary database is PostgreSQL 15. 
Redis is used for caching and session management. All services are containerized using Docker 
and orchestrated with Kubernetes. CI/CD is handled through GitHub Actions.

## API Guidelines
All APIs must follow RESTful conventions. Authentication uses JWT tokens with 1-hour expiry.
Rate limiting is set to 100 requests per minute per user. All responses must include proper 
HTTP status codes and error messages in JSON format.
"""

rag.add_document(tech_doc)

# حالا می‌توانیم سؤالات هر دو سند را بپرسیم!
answer, _ = rag.ask("What database does the company use?")
print(f"سؤال: What database does the company use?")
print(f"پاسخ: {answer}")

print()
answer, _ = rag.ask("What is the API rate limit?")
print(f"سؤال: What is the API rate limit?")
print(f"پاسخ: {answer}")

NameError: name 'get_embeddings' is not defined

## تمرین: RAG خودتان را بسازید

سعی کنید سند خودتان (هر متنی درباره موضوعی که می‌شناسید) را اضافه کنید و سؤالاتی درباره آن بپرسید.

In [ ]:
# تمرین: سند خودتان را اینجا اضافه کنید و سؤال بپرسید!
my_document = """
اینجا متن خودتان را قرار دهید...
"""

# مثال:
# rag.add_document(my_document)
# answer, sources = rag.ask("سؤال خود را اینجا بنویسید")
# print(answer)

NameError: name 'get_embeddings' is not defined

## خلاصه

در این جلسه یاد گرفتیم:
1. **RAG چیست** — تولید تقویت‌شده با بازیابی، ترکیب جستجو + تولید
2. **Embedding و جستجوی معنایی** — تبدیل متن به بردار برای یافتن اسناد مرتبط
3. **ساخت یک سیستم RAG** — تقسیم اسناد، ایندکس‌گذاری، بازیابی و تولید پاسخ
4. **مرزهای RAG** — تشخیص محدودیت‌ها و پاسخ «نمی‌دانم» در مواقع مناسب
5. **مقیاس‌پذیری** — افزودن آسان اسناد جدید به سیستم

**جلسه بعدی**: فراخوانی توابع (Function Calling) — به LLM قدرت اجرای عملیات واقعی بدهید!